<div style="background: linear-gradient(135deg, #050510 0%, #1a0530 35%, #0a2540 70%, #050510 100%);padding: 60px 40px; border-radius: 18px; text-align: center; margin-bottom: 10px; border: 1px solid #ff6b3544;"><h1 style="color: #ff6b35; font-family: 'Courier New', monospace; font-size: 2.8em;letter-spacing: 5px; margin: 0 0 12px 0; text-shadow: 0 0 40px #ff6b3566;">🎨 OpenCV DEVAM</h1><h2 style="color: #a8dadc; font-family: 'Courier New', monospace;font-size: 1.3em; margin: 0 0 18px 0; font-weight: 300;">9. Hafta — Aritmetik İşlemler, Bitwise, Maskeleme & Histogram</h2><p style="color: #6b7280; font-family: 'Courier New', monospace; font-size: 0.85em; margin: 0;">cv.add/subtract &bull; addWeighted &bull; Bitwise AND/OR/XOR &bull; Maske &bull; Histogram &bull; equalizeHist &bull; CLAHE</p></div>

---
## 📋 3 Ders Saati — Akış Planı

```
⏱️ 1. Ders (45 dk) — Aritmetik İşlemler
   • Saturation vs Modular aritmetik tuzağı
   • cv.add, cv.subtract, cv.multiply
   • cv.addWeighted ile görüntü harmanlama (alpha blending)
   • Parlaklık ve kontrast manipülasyonu

⏱️ 2. Ders (45 dk) — Bitwise & Maskeleme
   • Bitwise operatörler (AND, OR, XOR, NOT)
   • Maske oluşturma ve uygulama
   • Logo/Watermark ekleme
   • ROI ile maskelenmiş bölgeye çalışma

⏱️ 3. Ders (45 dk) — Histogram & Kontrast
   • calcHist ve görselleştirme
   • equalizeHist — histogram eşitleme
   • CLAHE — adaptif histogram eşitleme
   • convertScaleAbs — parlaklık/kontrast hızlı ayar
```

<div style="background: linear-gradient(135deg, #1a0530 0%, #0a2540 100%); padding: 20px 28px; border-radius: 12px; margin: 22px 0 10px 0; border-left: 5px solid #ff6b35;"><strong style="color: #ff6b35; font-size: 1.25em;">⏱️ 1. DERS SAATİ (45 dk)</strong><br><span style="color: #a8dadc; font-size: 0.95em;">Aritmetik İşlemler — Saturation, Alpha Blending, Parlaklık/Kontrast</span></div>

# 🧮 BÖLÜM 1: Görüntü Aritmetiği

## 1.1 Problem: Piksel Değeri Neden 200+100 = 44 Çıkıyor?

NumPy'da `uint8` dizilerde aritmetik işlem **modular** (wrap-around) yapılır. Yani 255'i geçen değerler en baştan sayılır: `255+1 = 0`, `200+100 = 300 % 256 = 44`.

Bu görüntü işlemede felaket! Beyaza yakın bir pikseli parlatınca karanlığa dönüşür.

In [ ]:
import numpy as np
import cv2 as cv

# İki piksel değeri
a = np.array([200], dtype=np.uint8)
b = np.array([100], dtype=np.uint8)

# NumPy + operatörü: MODULAR (tuzak!)
sonuc_numpy = a + b
print(f"NumPy + : {a[0]} + {b[0]} = {sonuc_numpy[0]}   ← YANLIŞ! (300%256=44)")

# OpenCV cv.add: SATURATION (doğru davranış)
sonuc_opencv = cv.add(a, b)
print(f"cv.add  : {a[0]} + {b[0]} = {sonuc_opencv[0]}   ← DOĞRU (255'i geçemez)")

# cv.subtract: 0'ın altına düşmez
c = np.array([50],  dtype=np.uint8)
d = np.array([100], dtype=np.uint8)
print(f"\ncv.subtract: {c[0]} - {d[0]} = {cv.subtract(c, d)[0]}  ← 0'da kalır")
print(f"NumPy -   : {c[0]} - {d[0]} = {(c - d)[0]}             ← YANLIŞ!")

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Saturation Aritmetiği — Donanım Kökeni:</b><br><br>Modern CPU'larda MMX, SSE, AVX komut setleri "saturated arithmetic" destekler. Bu komutlar 0 ve 255 sınırlarını otomatik uygular — tek bir işlemci komutunda. Intel'in MMX komut seti 1997'de tam olarak bu amaçla eklendi: multimedya işleme.<br><br>OpenCV'nin <code>cv.add</code> fonksiyonu arka planda bu SIMD komutlarını kullanır. Bu yüzden binlerce pikselde bile mikrosaniyeler içinde çalışır.<br><br><b>Kural:</b> Görüntü aritmetiğinde <b>asla</b> NumPy'nın <code>+</code>, <code>-</code>, <code>*</code> operatörlerini uint8 görüntülerde kullanmayın — ya <code>cv.add/subtract</code> kullanın, ya da önce <code>float32</code>'ye çevirin.</span></div>

## 1.2 Parlaklık ve Kontrast — Lineer Dönüşüm

Her piksel için basit formül:

$$\boxed{g(x,y) = \alpha \cdot f(x,y) + \beta}$$

- $\alpha$ (alpha): **kontrast** çarpanı (1.0 = aynı, >1 artar, <1 azalır)
- $\beta$ (beta): **parlaklık** farkı (0 = aynı, >0 artar, <0 azalır)

OpenCV'nin `cv.convertScaleAbs()` fonksiyonu bunu optimize şekilde yapar:

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Test görüntüsü: gradient (net fark görebilmek için)
resim = np.tile(np.arange(256, dtype=np.uint8), (200, 1))
resim = cv.cvtColor(resim, cv.COLOR_GRAY2BGR)

# Farklı alpha/beta kombinasyonları
sonuclar = [
    (resim,                                       "Orijinal (α=1, β=0)"),
    (cv.convertScaleAbs(resim, alpha=1.0, beta=50),  "Parlaklık +50 (α=1, β=50)"),
    (cv.convertScaleAbs(resim, alpha=1.0, beta=-50), "Parlaklık −50 (α=1, β=-50)"),
    (cv.convertScaleAbs(resim, alpha=1.5, beta=0),   "Kontrast ×1.5 (α=1.5, β=0)"),
    (cv.convertScaleAbs(resim, alpha=0.5, beta=0),   "Kontrast ×0.5 (α=0.5, β=0)"),
    (cv.convertScaleAbs(resim, alpha=1.5, beta=30),  "Kontrast ×1.5 + Parlaklık +30"),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
fig.suptitle("α × piksel + β — Parlaklık ve Kontrast Kontrolü", fontsize=13)
for ax, (g, b) in zip(axes.flat, sonuclar):
    ax.imshow(cv.cvtColor(g, cv.COLOR_BGR2RGB))
    ax.set_title(b, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>convertScaleAbs Neden Manuel Hesaplamadan Daha İyi?</b><br><br><code>sonuc = 1.5 * resim + 30</code> yazdığınızda:<br>1. <code>1.5 * resim</code> → float64 array oluşur (8× bellek)<br>2. <code>+ 30</code> → hâlâ float64<br>3. <code>uint8</code>'e geri dönüşümde taşma sorunu<br>4. 255 üstü değerler yanlış davranış<br><br><code>cv.convertScaleAbs()</code> bunların hepsini tek adımda, saturation ile, SIMD optimizasyonuyla halleder. Adından belli: "Convert + Scale + Absolute value".</span></div>

## 1.3 Görüntü Harmanlama — `cv.addWeighted()`

İki görüntüyü ağırlıklı toplamla karıştırma (alpha blending):

$$\boxed{g = \alpha \cdot f_1 + \beta \cdot f_2 + \gamma}$$

```python
cv.addWeighted(src1, alpha, src2, beta, gamma)
```

**Klasik kullanım:** $\alpha + \beta = 1$ olacak şekilde şeffaflık geçişi.

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# İki test görüntüsü — aynı boyutlu olmalı!
img1 = np.zeros((200, 400, 3), dtype=np.uint8)
cv.rectangle(img1, (20,20), (180,180), (50,50,200), -1)
cv.putText(img1, "GORUNTU 1", (200, 100), cv.FONT_HERSHEY_DUPLEX, 1.2, (255,255,255), 2)

img2 = np.zeros((200, 400, 3), dtype=np.uint8)
cv.circle(img2, (200, 100), 80, (50, 200, 50), -1)
cv.putText(img2, "GORUNTU 2", (50, 180), cv.FONT_HERSHEY_DUPLEX, 0.9, (255,255,0), 2)

# Farklı alpha değerleri ile harmanlama
alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
fig.suptitle("cv.addWeighted — α Değiştikçe Geçiş", fontsize=12)

for ax, alpha in zip(axes, alphas):
    beta = 1.0 - alpha
    karma = cv.addWeighted(img1, alpha, img2, beta, 0)
    ax.imshow(cv.cvtColor(karma, cv.COLOR_BGR2RGB))
    ax.set_title(f"α={alpha}, β={beta:.2f}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("α=0.0  → tamamen img2")
print("α=0.5  → ikisi eşit")
print("α=1.0  → tamamen img1")
print("α+β=1 → parlaklık sabit kalır")

<div style="background: #2d0020; border-left: 5px solid #f72585; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f72585; font-size: 1.08em;">🌍 Gerçek Dünya</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Alpha Blending'in Milyarlarca Kullanımı:</b><br><br><b>PowerPoint geçişleri:</b> Her "fade" geçişi addWeighted'dir.<br><b>Video editörleri (Premiere, DaVinci):</b> Iki klip arasındaki crossfade aynı formül.<br><b>Instagram filtreleri:</b> Filter preset'i = orijinal ile filtrelenmiş arasında α=0.6 harmanlama.<br><b>Oyunlar:</b> Duman, ateş, büyü efektleri alpha blending ile arka plana işlenir.<br><b>Tıbbi görüntüleme:</b> MRI + CT üst üste bindirme (multi-modal fusion).<br><br>Porter-Duff 1984 makalesi bu konuda "klasik" sayılır ve modern grafik pipeline'larının temelidir.</span></div>

## 1.4 Pratik Örnek: İki Görüntüyü Karışım Oranı ile Birleştirme

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Animasyon etkisi: 30 karede birinden diğerine geçiş
img1 = np.full((200, 400, 3), (200, 100, 50), dtype=np.uint8)   # turkuaz
img2 = np.full((200, 400, 3), (50, 100, 200), dtype=np.uint8)   # turuncu

# 9 adımda geçiş göster
fig, axes = plt.subplots(3, 3, figsize=(12, 6))
fig.suptitle("Cross-fade Animasyonu — addWeighted ile 9 Kare", fontsize=12)

for i, ax in enumerate(axes.flat):
    alpha = i / 8.0   # 0 → 1 arasında 9 değer
    karma = cv.addWeighted(img1, 1 - alpha, img2, alpha, 0)
    ax.imshow(cv.cvtColor(karma, cv.COLOR_BGR2RGB))
    ax.set_title(f"Kare {i+1}/9 (α={alpha:.2f})", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

<div style="background: linear-gradient(135deg, #1a0530 0%, #0a2540 100%); padding: 20px 28px; border-radius: 12px; margin: 22px 0 10px 0; border-left: 5px solid #ff6b35;"><strong style="color: #ff6b35; font-size: 1.25em;">⏱️ 2. DERS SAATİ (45 dk)</strong><br><span style="color: #a8dadc; font-size: 0.95em;">Bitwise Operatörler, Maskeleme, Logo/Watermark Ekleme</span></div>

# 🔲 BÖLÜM 2: Bitwise (Bit Düzeyinde) İşlemler

## 2.1 Bit Düzeyinde Düşünmek

Bir pikselin değeri olarak tek bir sayı (ör. 200) olsa da bu sayı **binary** olarak 8 bit ile ifade edilir: `200 = 11001000`. Bitwise operatörler bu bitleri tek tek işler.

```
AND (&):   Her iki bit 1 ise sonuç 1           1100 & 1010 = 1000
OR  (|):   Herhangi biri 1 ise sonuç 1          1100 | 1010 = 1110
XOR (^):   Tam olarak biri 1 ise sonuç 1        1100 ^ 1010 = 0110
NOT (~):   Tersini al (255 − piksel)            1100 → 0011
```

| Operatör | OpenCV Fonksiyonu | Tipik Kullanım |
|----------|-------------------|----------------|
| AND | `cv.bitwise_and` | Maske uygulama (belirli bölgeyi koru) |
| OR  | `cv.bitwise_or`  | İki maskeyi birleştir |
| XOR | `cv.bitwise_xor` | Fark bulma, şifreleme |
| NOT | `cv.bitwise_not` | Negatif görüntü / maske ters çevirme |

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Bitwise İşlemlerin İlginç Bir Kullanımı — XOR Şifreleme:</b><br><br>İkinci Dünya Savaşı'nda Alman Enigma'sının kırılmasından sonra 1917'de Gilbert Vernam <b>XOR tabanlı şifreleme</b> geliştirdi. Aynı anahtarı iki kez XOR'larsanız orijinal veriye dönersiniz:<br><code>mesaj XOR anahtar = şifreli</code><br><code>şifreli XOR anahtar = mesaj</code><br><br>Görüntü işlemede XOR ile "gizli görüntü" üretebilirsiniz: iki rastgele görüntü XOR edilirse rastgele görünen bir sonuç çıkar, ama ikisinden birini XOR ederseniz diğeri ortaya çıkar. Bu tekniğe <b>visual cryptography</b> denir ve 1994'te Naor & Shamir tarafından sistematize edildi (Shamir RSA'nın "S" harfidir).</span></div>

## 2.2 Bitwise Operatörlerin Görsel Etkisi

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# İki basit geometrik şekil hazırla
img1 = np.zeros((300, 300), dtype=np.uint8)
cv.rectangle(img1, (50, 50), (200, 200), 255, -1)   # beyaz dikdörtgen

img2 = np.zeros((300, 300), dtype=np.uint8)
cv.circle(img2, (180, 180), 80, 255, -1)            # beyaz çember

# Tüm bitwise işlemleri uygula
and_sonuc = cv.bitwise_and(img1, img2)
or_sonuc  = cv.bitwise_or (img1, img2)
xor_sonuc = cv.bitwise_xor(img1, img2)
not_sonuc = cv.bitwise_not(img1)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle("Bitwise İşlemlerin Geometrik Yorumu", fontsize=13)

imgs = [
    (img1,      "Görüntü 1 (Dikdörtgen)"),
    (img2,      "Görüntü 2 (Çember)"),
    (and_sonuc, "AND → Ortak alan"),
    (or_sonuc,  "OR  → Birleşim"),
    (xor_sonuc, "XOR → Simetrik fark"),
    (not_sonuc, "NOT → Negatif"),
]

for ax, (g, title) in zip(axes.flat, imgs):
    ax.imshow(g, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Bitwise İşlemler = Küme İşlemleri!</b><br><br>İkili görüntülerde bitwise operatörler <b>kümelerin mantıksal işlemlerine</b> eşittir:<br><br>• <code>AND</code> (∩) → Kesişim: sadece iki kümede de olan elemanlar<br>• <code>OR</code> (∪) → Birleşim: ikisinden birinde olan tüm elemanlar<br>• <code>XOR</code> (△) → Simetrik fark: yalnızca birinde olan elemanlar<br>• <code>NOT</code> (ᶜ) → Tümleyen: kümeye ait olmayan elemanlar<br><br>Bu yüzden bitwise işlemler görüntü segmentasyonu, nesne sınırları birleştirme, maske kombinasyonu için matematiksel olarak <b>doğal araçtır</b>.</span></div>

## 2.3 Maskeleme — En Kritik Görüntü İşleme Tekniği

**Maske** nedir? İkili (0 ve 255) bir görüntü. Hangi piksellerin "geçerli" olduğunu belirtir.

```
maske[i,j] = 255  →  o piksel işlenir/korunur
maske[i,j] = 0    →  o piksel atlanır/silinir
```

`cv.bitwise_and(src1, src2, mask=maske)` → yalnızca maskenin 255 olduğu yerlerde işlem yapar.

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Renkli test görüntüsü — manzara simülasyonu
resim = np.zeros((300, 400, 3), dtype=np.uint8)
resim[:150, :] = (200, 180, 100)    # gökyüzü (mavi-açık)
resim[150:, :] = (50, 120, 50)      # yeşillik
cv.circle(resim, (320, 50), 25, (0, 255, 255), -1)   # güneş
cv.rectangle(resim, (150, 180), (250, 280), (30, 60, 100), -1)  # ev
cv.putText(resim, "MANZARA", (140, 30), cv.FONT_HERSHEY_DUPLEX, 0.8, (255,255,255), 2)

# Dairesel maske oluştur (merkezine bakacağız)
maske = np.zeros(resim.shape[:2], dtype=np.uint8)
cv.circle(maske, (200, 150), 120, 255, -1)   # içi dolu beyaz daire

# Maskelenmiş sonuç
sonuc = cv.bitwise_and(resim, resim, mask=maske)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(cv.cvtColor(resim, cv.COLOR_BGR2RGB))
axes[0].set_title("Orijinal Görüntü", fontsize=10)
axes[0].axis("off")

axes[1].imshow(maske, cmap="gray")
axes[1].set_title("Dairesel Maske\n(255=görünür, 0=gizli)", fontsize=10)
axes[1].axis("off")

axes[2].imshow(cv.cvtColor(sonuc, cv.COLOR_BGR2RGB))
axes[2].set_title("bitwise_and(resim, resim, mask=maske)", fontsize=10)
axes[2].axis("off")

plt.tight_layout()
plt.show()

<div style="background: #2d2200; border-left: 5px solid #f9c74f; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f9c74f; font-size: 1.08em;">⚠️ Kritik Nokta</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>cv.bitwise_and'de Neden Aynı Görüntüyü İki Kez Veriyoruz?</b><br><br><code>cv.bitwise_and(resim, resim, mask=maske)</code><br><br>Sezgisel olarak "maskelensin" diye tek görüntü yeterli gibi görünüyor. Ama <code>bitwise_and</code> bir <b>ikili operatördür</b>, iki operand gerekir.<br><br><code>resim AND resim = resim</code> (bir sayının kendisiyle AND'i o sayıdır). Sonra mask parametresi "sadece şu piksellerde işlem yap" der. Net sonuç: maskedeki 0 olan yerler siyaha (0) düşer, 255 olan yerler orijinali korur.<br><br>Alternatif: <code>resim[maske == 0] = 0</code> — NumPy ile. Ama <code>cv.bitwise_and</code> tüm kanallara otomatik uygulanır, daha temizdir.</span></div>

## 2.4 Uygulama: Logo/Watermark Ekleme

Klasik örnek: şeffaf bir logoyu ana görüntüye koymak. 5 adımda yapılır:

1. Logo'nun görüntüye yerleştirileceği bölgeyi (ROI) al
2. Logo'yu gri'ye çevir ve eşikleyerek **siyah-beyaz maske** yap
3. Maskenin tersini al (logo dışındaki alanlar)
4. Ana görüntüden logo bölgesi siyahlanır (`bitwise_and` + ters maske)
5. Logo + siyahlanmış ana görüntüyü birleştir (`bitwise_or`)

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# ── Ana görüntü hazırla (400×600 manzara simülasyonu) ─────────────
ana = np.zeros((400, 600, 3), dtype=np.uint8)
ana[:, :] = (120, 180, 220)   # gökyüzü
ana[250:, :] = (50, 100, 50)  # yeşil alan
cv.circle(ana, (500, 60), 30, (100, 220, 230), -1)   # güneş
for i in range(0, 600, 80):   # ağaçlar
    cv.circle(ana, (i, 300), 25, (30, 80, 30), -1)

# ── Logo hazırla (beyaz zeminli, siyah yazılı) ────────────────────
logo = np.full((80, 200, 3), 255, dtype=np.uint8)   # beyaz arka plan
cv.putText(logo, "OPENCV", (10, 50), cv.FONT_HERSHEY_DUPLEX, 1.3, (0, 0, 0), 3)
cv.rectangle(logo, (0, 0), (199, 79), (0, 0, 0), 2)

# ── Logo'yu sol üst köşeye yerleştireceğiz ────────────────────────
h, w = logo.shape[:2]
roi = ana[10:10+h, 10:10+w]    # aynı boyutta ROI

# ── Adım 1: Logo'yu gri + eşikle → siyah-beyaz maske ──────────────
logo_gri = cv.cvtColor(logo, cv.COLOR_BGR2GRAY)
ret, maske = cv.threshold(logo_gri, 200, 255, cv.THRESH_BINARY)
# maske: beyaz zemin=255, yazı=0

maske_ters = cv.bitwise_not(maske)
# maske_ters: yazı=255, zemin=0

# ── Adım 2: ROI'den yazı bölümünü siyahla (yazı kaldıracağımız yer) ───
ana_temiz = cv.bitwise_and(roi, roi, mask=maske)
# ROI'den yazı alanı siyah oldu, zemin orijinal kaldı

# ── Adım 3: Logo'dan sadece yazı bölümünü al ─────────────────────────
logo_yazi = cv.bitwise_and(logo, logo, mask=maske_ters)
# Logo'nun zemini siyah oldu, yazı kaldı

# ── Adım 4: İkisini birleştir ───────────────────────────────────────
sonuc = cv.add(ana_temiz, logo_yazi)
ana[10:10+h, 10:10+w] = sonuc

# Tüm adımları göster
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
parçalar = [
    (logo,          "1. Logo (Beyaz zeminli)"),
    (maske,         "2. Maske (eşikleme ile)"),
    (maske_ters,    "3. Ters maske"),
    (ana_temiz,     "4. ROI'den yazı kesildi"),
    (logo_yazi,     "5. Sadece yazı alındı"),
    (ana,           "6. Sonuç: Logo eklendi"),
]
for ax, (g, t) in zip(axes.flat, parçalar):
    cmap = "gray" if len(g.shape) == 2 else None
    img = g if cmap else cv.cvtColor(g, cv.COLOR_BGR2RGB)
    ax.imshow(img, cmap=cmap)
    ax.set_title(t, fontsize=9)
    ax.axis("off")
plt.suptitle("Logo Ekleme — 6 Adım", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 Watermarking — Dijital Dünyanın Güvenliği</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Watermark (filigran) aslında 13. yüzyıl İtalyan kağıt üreticilerinden gelir. Kağıdın içine ince tel süzgeçler yerleştirilerek üretim sırasında kağıtta görünmez desenler oluştururdular.<br><br><b>Dijital watermarking:</b> 1990'larda internetin yaygınlaşmasıyla telif hakkı koruması için geliştirildi. İki tür:<br>• <b>Görünür watermark:</b> Bizim yukarıdaki örnek gibi. Stock photo siteleri (Shutterstock, Getty Images) bunu kullanır.<br>• <b>Görünmez watermark:</b> DCT/DWT alanında pikselleri minimal değiştirerek çıplak gözle görünmeyen ama yazılımla tespit edilebilen işaret. Netflix, Disney+ her kullanıcıya özel görünmez watermark basar — video sızdırılırsa kimin hesabından geldiği anlaşılır.</span></div>

<div style="background: linear-gradient(135deg, #1a0530 0%, #0a2540 100%); padding: 20px 28px; border-radius: 12px; margin: 22px 0 10px 0; border-left: 5px solid #ff6b35;"><strong style="color: #ff6b35; font-size: 1.25em;">⏱️ 3. DERS SAATİ (45 dk)</strong><br><span style="color: #a8dadc; font-size: 0.95em;">Histogram, Histogram Eşitleme, CLAHE</span></div>

# 📊 BÖLÜM 3: Histogram — Görüntünün Parmak İzi

## 3.1 Histogram Nedir?

Bir görüntünün histogramı, her piksel değerinin kaç kez göründüğünü gösterir.

```
X ekseni: piksel değerleri (0-255)
Y ekseni: o değerdeki piksel sayısı

Karanlık görüntü:        Parlak görüntü:        Düşük kontrast:
  │                       │                       │
  │ ▆▄                    │         ▄▆           │       ▆▄
  │ ▆▆▆                   │         ▆▆▆          │       ▆▆
  └──────────             └──────────             └──────────
  0         255           0         255           0         255
  
(sola toplanmış)        (sağa toplanmış)       (dar bir bölgede)
```

```python
hist = cv.calcHist([img], [kanal], mask, [bin_sayisi], [min, max])
```

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Farklı kontrastlarda 3 test görüntüsü
base = np.tile(np.linspace(80, 160, 400, dtype=np.uint8), (300, 1))
# orta parlaklıkta, düşük kontrast
karanlik = np.clip(base - 50, 0, 255).astype(np.uint8)
parlak   = np.clip(base + 70, 0, 255).astype(np.uint8)
yuksek   = np.clip((base.astype(int) - 120) * 3 + 128, 0, 255).astype(np.uint8)

gorsel_listesi = [
    (base,     "Düşük Kontrast"),
    (karanlik, "Karanlık"),
    (parlak,   "Parlak"),
    (yuksek,   "Yüksek Kontrast"),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle("Görüntü Türlerine Göre Histogram Şekilleri", fontsize=13)

for col, (img, title) in enumerate(gorsel_listesi):
    axes[0, col].imshow(img, cmap="gray", vmin=0, vmax=255)
    axes[0, col].set_title(title, fontsize=10)
    axes[0, col].axis("off")

    hist = cv.calcHist([img], [0], None, [256], [0, 256])
    axes[1, col].fill_between(range(256), hist.ravel(), alpha=0.7, color="#ff6b35")
    axes[1, col].set_xlim(0, 255)
    axes[1, col].set_title("Histogram", fontsize=9)
    axes[1, col].set_xlabel("Piksel Değeri")
    axes[1, col].set_ylabel("Frekans")
    axes[1, col].grid(alpha=0.3)

plt.tight_layout()
plt.show()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>calcHist Parametrelerinin Gerçek Anlamı:</b><br><br><code>cv.calcHist([img], [0], None, [256], [0, 256])</code><br><br>• <b>[img]</b> — liste içinde tek görüntü (birden fazla da olabilir)<br>• <b>[0]</b> — hesaplanacak kanal (gri tonlamada 0, BGR'de 0=B, 1=G, 2=R)<br>• <b>None</b> — maske (sadece belirli bölgenin histogramı isteniyorsa)<br>• <b>[256]</b> — bin sayısı (256 = her değer için ayrı; 64 = 4'er gruplanmış)<br>• <b>[0, 256]</b> — değer aralığı<br><br><b>Renkli görüntü histogramı:</b> Her kanal için ayrı hesaplanır:<br><code>hist_b = cv.calcHist([img], [0], None, [256], [0, 256])</code><br><code>hist_g = cv.calcHist([img], [1], None, [256], [0, 256])</code><br><code>hist_r = cv.calcHist([img], [2], None, [256], [0, 256])</code></span></div>

## 3.2 Renkli Histogram ve Maskeli Histogram

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Renkli test görüntüsü
resim = np.zeros((300, 400, 3), dtype=np.uint8)
resim[:150, :200] = (200, 50, 50)   # mavi-ağırlıklı
resim[:150, 200:] = (50, 200, 50)   # yeşil-ağırlıklı
resim[150:, :200] = (50, 50, 200)   # kırmızı-ağırlıklı
resim[150:, 200:] = (200, 200, 200) # gri

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].imshow(cv.cvtColor(resim, cv.COLOR_BGR2RGB))
axes[0].set_title("4 Bölgeli Renkli Görüntü")
axes[0].axis("off")

# Üç kanal için ayrı histogram
renkler_kanal = [("Blue", "b"), ("Green", "g"), ("Red", "r")]
for kanal_idx, (isim, mpl_color) in enumerate(renkler_kanal):
    hist = cv.calcHist([resim], [kanal_idx], None, [256], [0, 256])
    axes[1].plot(hist, color=mpl_color, label=isim, linewidth=1.5)

axes[1].set_xlim(0, 255)
axes[1].set_xlabel("Piksel Değeri")
axes[1].set_ylabel("Frekans")
axes[1].set_title("BGR Kanal Histogramları")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3.3 Histogram Eşitleme — `cv.equalizeHist()`

Histogram eşitleme, dar bir aralıkta yığılmış piksel değerlerini **tüm 0-255 aralığına eşit dağıtarak** kontrastı artırır.

```
Önce:                    Sonra:
│                          │
│   ████                   │ ▆  ▆  ▆  ▆  ▆
│  ██████                  │ ▆  ▆  ▆  ▆  ▆
└──────────                └──────────────
0  80  160  255            0      128     255

(dar, düşük kontrast)     (geniş, yüksek kontrast)
```

**Matematiksel olarak:** Kümülatif dağılım fonksiyonu (CDF) kullanılarak her piksel değeri yeni bir değere eşlenir. Bu doğrusal olmayan bir dönüşümdür!

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Histogram Eşitleme'nin Tarihi — 1974:</b><br><br>R.E. Woods ve R.C. Gonzalez'in meşhur "Digital Image Processing" kitabında (ilk baskı 1977) detaylıca işlenen teknik aslında 1970'lerin başında NASA'nın uzay görüntülerini iyileştirmesi için geliştirildi.<br><br>Apollo ve Viking programlarında Ay'dan ve Mars'tan gelen görüntüler düşük kontrastlı, karanlık yerlerdi — atmosfer olmayan bir gezegende gölgeler çok sert, detaylar kayıp oluyordu. NASA bu sorunu çözmek için histogram eşitleme ile gamma düzeltmeyi birleştirdi.<br><br>Bugün radyografi (röntgen), mamografi, uydu görüntüleri, gece görüşü — hepsinde temel ön işleme adımıdır. Tek satır kodla büyük etki yaratır.</span></div>

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Düşük kontrastlı test görüntüsü
resim = np.zeros((300, 400), dtype=np.uint8)
# Değerleri 80-160 arasında sıkıştır
resim = np.tile(np.linspace(80, 160, 400, dtype=np.uint8), (300, 1))
# Biraz varyasyon ekle
np.random.seed(0)
resim = np.clip(resim + np.random.normal(0, 15, resim.shape), 0, 255).astype(np.uint8)

# Histogram eşitleme uygula
esitlenmis = cv.equalizeHist(resim)

# Sonuçları ve histogramları göster
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle("Histogram Eşitleme — cv.equalizeHist()", fontsize=13, fontweight="bold")

axes[0, 0].imshow(resim, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Orijinal (Düşük Kontrast)")
axes[0, 0].axis("off")

axes[0, 1].imshow(esitlenmis, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("Histogram Eşitlenmiş")
axes[0, 1].axis("off")

# Histogramlar
hist_orig = cv.calcHist([resim],      [0], None, [256], [0, 256])
hist_eq   = cv.calcHist([esitlenmis], [0], None, [256], [0, 256])

axes[1, 0].fill_between(range(256), hist_orig.ravel(), color="#e74c3c", alpha=0.7)
axes[1, 0].set_xlim(0, 255)
axes[1, 0].set_title("Orijinal Histogram (80-160 arası yığılma)")
axes[1, 0].set_xlabel("Piksel Değeri")
axes[1, 0].grid(alpha=0.3)

axes[1, 1].fill_between(range(256), hist_eq.ravel(), color="#27ae60", alpha=0.7)
axes[1, 1].set_xlim(0, 255)
axes[1, 1].set_title("Eşitlenmiş Histogram (0-255 dağılmış)")
axes[1, 1].set_xlabel("Piksel Değeri")
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Orijinal:    min={resim.min()}, max={resim.max()}, std={resim.std():.1f}")
print(f"Eşitlenmiş:  min={esitlenmis.min()}, max={esitlenmis.max()}, std={esitlenmis.std():.1f}")

## 3.4 CLAHE — Adaptif Histogram Eşitleme

`equalizeHist` **global**dir — tüm görüntü için aynı dönüşüm uygulanır. Bu, bazı bölgelerde aşırı kontrast, diğerlerinde detay kaybına yol açabilir.

**CLAHE** (Contrast Limited Adaptive Histogram Equalization):
- Görüntüyü küçük bloklara böler (`tileGridSize`)
- Her bloğa ayrı histogram eşitleme uygular
- Aşırı kontrastı sınırlar (`clipLimit`)
- Blok sınırlarını bilineer interpolasyonla yumuşatır

```python
clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
sonuc = clahe.apply(gri_goruntu)
```

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Eşit olmayan aydınlatmalı test görüntüsü
h, w = 300, 400
resim = np.zeros((h, w), dtype=np.uint8)

# Sol yarı karanlık, sağ yarı parlak (gerçek hayattaki ışık eşitsizliği)
for i in range(h):
    for j in range(w):
        resim[i, j] = int(100 + 100 * (j/w) + 50 * np.sin(i*0.05) * np.cos(j*0.05))
resim = np.clip(resim, 0, 255).astype(np.uint8)

# Bazı detaylar ekle
cv.putText(resim, "DETAY-A", (30, 100),  cv.FONT_HERSHEY_DUPLEX, 0.8, 60,  2)
cv.putText(resim, "DETAY-B", (240, 250), cv.FONT_HERSHEY_DUPLEX, 0.8, 220, 2)

# Global equalize
global_eq = cv.equalizeHist(resim)

# CLAHE — farklı clipLimit değerleriyle
clahe1 = cv.createCLAHE(clipLimit=1.0, tileGridSize=(8, 8)).apply(resim)
clahe2 = cv.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(resim)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("equalizeHist vs CLAHE — Eşit Olmayan Aydınlatma", fontsize=13, fontweight="bold")

for ax, (g, t) in zip(axes.flat, [
    (resim,     "Orijinal (sol karanlık, sağ parlak)"),
    (global_eq, "cv.equalizeHist (global)"),
    (clahe1,    "CLAHE (clipLimit=1.0)"),
    (clahe2,    "CLAHE (clipLimit=3.0)"),
]):
    ax.imshow(g, cmap="gray", vmin=0, vmax=255)
    ax.set_title(t, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>CLAHE Parametrelerini Ne Zaman Değiştirmeli?</b><br><br><b>clipLimit:</b> 1.0-4.0 tipik aralık.<br>• Düşük (1.0-2.0) → doğal görünüm, yumuşak<br>• Yüksek (3.0-4.0+) → agresif kontrast artışı, gürültü belirginleşebilir<br><br><b>tileGridSize:</b> (8, 8) varsayılan, çoğunlukla iyi.<br>• Küçük (4×4) → daha yerel, ince detay<br>• Büyük (16×16) → daha global, yumuşak<br><br><b>Renkli görüntülerde CLAHE:</b> Doğrudan BGR'a uygulamayın! Renk bozulur. Önce LAB uzayına çevirin, yalnızca L kanalına uygulayın, geri dönün:<br><code>lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)<br>lab[:,:,0] = clahe.apply(lab[:,:,0])<br>sonuc = cv.cvtColor(lab, cv.COLOR_LAB2BGR)</code></span></div>

<div style="background: #2d0020; border-left: 5px solid #f72585; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f72585; font-size: 1.08em;">🌍 Gerçek Dünya</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>CLAHE'nin Gerçek Dünyadaki Hayat Kurtaran Uygulamaları:</b><br><br><b>Tıbbi radyografi:</b> Meme mamografisinde küçük tümörler ancak CLAHE ile belirgin hale gelir. DICOM görüntüleme yazılımlarının hepsi CLAHE destekler.<br><br><b>Akıllı telefon kameralar:</b> HDR modu ve gece modunda CLAHE türevleri kullanılır. Google Pixel'in "Night Sight" özelliği adaptif histogram + çoklu kare birleştirmesinin modern bir uygulamasıdır.<br><br><b>Endoskopi:</b> Mide-bağırsak endoskopisinde düşük ışıkta çekilen görüntüler CLAHE ile iyileştirilerek hastalık teşhisi kolaylaştırılır.<br><br><b>Uydu görüntüleri:</b> NASA'nın Mars rover'ları CLAHE ile gölgelerdeki detayları ortaya çıkarır — ışık bire bir olmayan yerlerdeki jeolojik detaylar.</span></div>

---
# 🏛️ BÖLÜM 4: Üniversite Seviyesi Alıştırmalar

**Cevaplar verilmemiştir.**

---
## ❓ Soru 1 — Fotoğraf Karşılaştırma: 10 Farkı Bul

**Zorluk: ⭐⭐⭐☆☆**

İki görsel olarak neredeyse aynı olan fotoğraf arasındaki farkları otomatik tespit eden ve farklılıkları kırmızı dikdörtgenlerle işaretleyen bir program yazın.

### Görev:
1. İki görüntüyü `cv.absdiff()` veya `cv.bitwise_xor` ile karşılaştır
2. Fark görüntüsünü gri'ye çevir, Gaussian blur uygula
3. Eşikleme ile farkları ikili görüntüye çevir
4. Morfolojik işlemler yerine `cv.connectedComponents` ile bağlı bileşenleri bul (her fark bölgesi bir bileşen)
5. Her bileşenin bounding box'ını orijinal görüntülere çiz
6. Toplam fark sayısını ve fark oranını (%) yazdır

### Düşünmeniz Gereken Sorular:
- absdiff ile bitwise_xor farkı nedir? Hangisi hangi tür için daha uygun?
- Çok küçük farkları (gürültü) nasıl eliyorsunuz?
- JPEG sıkıştırma artefaktları yanlış pozitif üretebilir — nasıl azaltılır?

```python
import cv2 as cv
import numpy as np

def farklari_bul(img1, img2, min_alan=50):
    # TODO
    pass
```

---
## ❓ Soru 2 — Dijital Watermarking Sistemi

**Zorluk: ⭐⭐⭐⭐☆**

Hem görünür hem **görünmez (subtle) watermark** ekleyebilen bir sistem tasarlayın.

### Görev:
1. **Görünür watermark:** Logo'yu belirli şeffaflıkla addWeighted ile ekle
2. **Görünmez watermark (LSB):** Watermark görüntüsünün her pikselini 1-bit'e dönüştür, ana görüntünün mavi kanalının son bitine göm
3. **Ters işlem:** Gömülü watermark'ı geri çıkaran fonksiyon
4. **Robustness testi:** JPEG sıkıştırmadan sonra watermark hâlâ çıkarılabiliyor mu? Quality 50, 75, 90 için test et
5. **PSNR hesabı:** Orijinal vs watermarked görüntü arasındaki PSNR ölç

### Düşünmeniz Gereken Sorular:
- LSB watermarking neden JPEG'e karşı kırılgandır?
- DCT tabanlı watermarking nasıl daha robust olur?
- Netflix'in her kullanıcıya özel görünmez watermark tekniği nasıl çalışıyor olabilir?

```python
import cv2 as cv
import numpy as np

def gorunur_watermark(ana, logo, alpha=0.3):
    # TODO
    pass

def gorunmez_watermark_goz(ana, watermark):
    """LSB steganografi ile watermark gömer."""
    # TODO
    pass

def watermark_cikart(watermarked):
    """Gömülü watermark'ı geri çıkarır."""
    # TODO
    pass
```

---
## ❓ Soru 3 — Otomatik Fotoğraf İyileştirici

**Zorluk: ⭐⭐⭐⭐☆**

Bir fotoğrafı analiz edip (karanlık mı, düşük kontrast mı, renkleri bozuk mu) uygun iyileştirmeyi otomatik uygulayan bir sistem yazın.

### Görev:
1. Görüntünün histogramını analiz et:
   - Ortalama parlaklık < 80 → "karanlık" etiketi
   - Standart sapma < 40 → "düşük kontrast" etiketi
   - RGB kanal ortalamaları arasında 30+ fark → "renk dengesiz"
2. Her sorun için farklı düzeltme stratejisi:
   - Karanlık → gamma < 1 düzeltme (gamma=0.6)
   - Düşük kontrast → CLAHE (clipLimit=2.0)
   - Renk dengesiz → gri dünya varsayımı ile white balance
3. Düzeltilmiş görüntüleri ve analiz raporunu yan yana göster
4. Öncesi/sonrası için aynı histogramı çiz

### Düşünmeniz Gereken Sorular:
- Gamma düzeltme matematiği nedir? Neden 2.2 "standart" sayılır?
- "Gray world" varsayımı her zaman doğru çalışır mı? Ne zaman başarısız olur?
- Instagram filtreleri bu mantığı nasıl genişletir?

```python
import cv2 as cv
import numpy as np

def goruntu_analiz(img):
    """Görüntünün sorunlarını tespit eder, etiket listesi döndürür."""
    pass

def gamma_duzeltme(img, gamma):
    """g(x) = 255 × (f(x)/255)^gamma"""
    pass

def otomatik_iyilestir(img):
    """Sorunları tespit edip uygun düzeltmeyi uygular."""
    pass
```

---
## ❓ Soru 4 — Yeşil Ekran (Chroma Key) 2.0 — Bitwise ile

**Zorluk: ⭐⭐⭐⭐☆**

7. haftada HSV maskeleme ile chroma key yapmıştık. Şimdi aynı işlemi **yalnızca bitwise ve aritmetik** operatörler kullanarak daha verimli yapalım.

### Görev:
1. Ön plan (yeşil ekranlı kamera) ve arka plan (manzara) görüntüsünü aynı boyuta getir
2. HSV'de yeşil maskesi oluştur
3. Maskenin tersini al (maske_ters = insanı gösteriyor)
4. `bitwise_and(foreground, mask=maske_ters)` → yalnızca insan kalsın
5. `bitwise_and(background, mask=maske)` → yalnızca arka planın yeşil kısmı kalsın
6. İki sonucu `cv.add` ile birleştir
7. **Bonus:** Kenar yumuşatma için maskeyi Gaussian blur uygula, sonra `addWeighted` ile birleştir

### Düşünmeniz Gereken Sorular:
- Sert maskeyle yumuşak maske (blurred) arasında görsel fark ne? Edge spillover nedir?
- Yeşil saçak (green spill) problemi nedir? Nasıl giderilir?
- Sinemadaki profesyonel chroma key yazılımları hangi ek teknikleri kullanır?

```python
import cv2 as cv
import numpy as np

def chroma_key_v2(foreground, background,
                   lower=(35, 50, 50), upper=(85, 255, 255)):
    """Yalnızca bitwise + add ile chroma key."""
    # TODO
    pass
```

---
## ❓ Soru 5 — Histogram Matching: Bir Fotoğrafın Stilini Transfer Et

**Zorluk: ⭐⭐⭐⭐⭐**

İki fotoğrafı girdi al:
- **Kaynak fotoğraf:** düzeltmek istediğiniz fotoğraf
- **Referans fotoğraf:** renk tonunu/atmosferini beğendiğiniz fotoğraf

Kaynak fotoğrafın histogramını referans fotoğrafın histogramına eşleyerek renk karakteristiğini transfer edin.

### Görev:
1. Her iki görüntünün kümülatif dağılım fonksiyonunu (CDF) hesapla
2. Her kaynak piksel değeri için:
   - Kaynak CDF'de o değerin yüzdesini bul (örn. 60. yüzdelik)
   - Referans CDF'de aynı yüzdelikteki değeri bul (örn. ref'te 60. yüzdelik = 145)
   - Kaynak pikseli 145'e dönüştür
3. Bu işlemi BGR her kanal için ayrı yap
4. LAB uzayında yapıp L'yi koruyarak denemek daha doğal sonuç verir — test et
5. Matplotlib ile: 2×3 grid (kaynak, referans, sonuç + 3 histogram)

### Düşünmeniz Gereken Sorular:
- Histogram matching ile neural style transfer arasındaki felsefi fark ne?
- Bu yöntem "yaz ambiyansı → sonbahar ambiyansı" gibi büyük değişimlerde neden başarısız olur?
- Adobe Lightroom'un "Reference" editing modu bu matematiği mi kullanıyor?

```python
import cv2 as cv
import numpy as np

def histogram_matching(kaynak, referans):
    """
    kaynak'ın histogramını referans'ın histogramına eşler.
    Her BGR kanalı için ayrı hesaplar.
    """
    # TODO: CDF hesapla, eşleme tablosu oluştur, uygula
    pass
```

---
<div style="background: linear-gradient(135deg, #050510 0%, #1a0530 40%, #0a2540 100%);padding: 38px 42px; border-radius: 18px; text-align: center; margin-top: 20px; border: 1px solid #ff6b3544;"><h2 style="color: #ff6b35; font-family: 'Courier New', monospace; margin: 0 0 22px 0;">🎯 9. Hafta Özeti</h2><div style="color: #cdd6f4; text-align: left; max-width: 700px; margin: 0 auto; line-height: 2.2;"><b>⏱️ Ders 1 — Aritmetik:</b><br>✅ NumPy <code>+</code> modular (tuzak), <code>cv.add</code> saturation<br>✅ <code>cv.convertScaleAbs(α, β)</code> — parlaklık/kontrast<br>✅ <code>cv.addWeighted</code> — alpha blending, crossfade<br><br><b>⏱️ Ders 2 — Bitwise & Maskeleme:</b><br>✅ AND/OR/XOR/NOT — küme işlemleri olarak<br>✅ <code>bitwise_and(img, img, mask=m)</code> — neden iki kez img?<br>✅ Logo/watermark ekleme — 6 adımlı pipeline<br><br><b>⏱️ Ders 3 — Histogram:</b><br>✅ <code>cv.calcHist</code> — görüntünün parmak izi<br>✅ <code>cv.equalizeHist</code> — global histogram eşitleme<br>✅ <code>cv.createCLAHE</code> — adaptif, eşit olmayan aydınlatma için<br>✅ clipLimit ve tileGridSize parametreleri<br>✅ Renkli CLAHE → LAB uzayında L kanalına uygula</div><p style="color: #6b7280; margin: 22px 0 0 0; font-family: 'Courier New', monospace;">10. Hafta → Affine & Perspektif Dönüşüm + Çizim Fonksiyonları 📐</p></div>